# Running AlphaFold

## Make sure imports work

In [1]:
import os, sys, json, pathlib, time
import numpy as np
import torch
import matplotlib.pyplot as plt

print("python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("cuda avail ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device     ", torch.cuda.get_device_name(0))
    print("vram (GB)  ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

import openfold
print("openfold   ", pathlib.Path(openfold.__file__).parent)
import attn_core_inplace_cuda
print("cuda kernel  OK")

python      3.10.20
torch       2.14.0+cu130
cuda avail  True
device      NVIDIA GeForce RTX 4060 Laptop GPU
vram (GB)   8.2
openfold    /home/caitlinlewis/repos/openfold/openfold
cuda kernel  OK


## Get protein info

In [2]:
import requests
from pathlib import Path

ACCESSIONS = {
    "myoglobin":  "P02185",  # sperm whale Mb, the 1MBN classic
    "GFP":        "P42212",  # Aequorea victoria avGFP
    "TIM":        "P00940",  # chicken triosephosphate isomerase, 1TIM
    "ubiquitin":  "P0CG48",  # human polyubiquitin-C
    "calmodulin": "P0DP23",  # human CALM1
}

out = Path("work/fasta"); out.mkdir(exist_ok=True)

for name, acc in ACCESSIONS.items():
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=30)
    r.raise_for_status()
    header, *seq = r.text.strip().split("\n")
    seq = "".join(seq)
    print(f"{name:12s} {acc}  len={len(seq):4d}  {header[:70]}")
    (out / f"{name}.fasta").write_text(f">{name}\n{seq}\n")

myoglobin    P02185  len= 154  >sp|P02185|MYG_PHYMC Myoglobin OS=Physeter macrocephalus OX=9755 GN=MB
GFP          P42212  len= 238  >sp|P42212|GFP_AEQVI Green fluorescent protein OS=Aequorea victoria OX
TIM          P00940  len= 248  >sp|P00940|TPIS_CHICK Triosephosphate isomerase OS=Gallus gallus OX=90
ubiquitin    P0CG48  len= 685  >sp|P0CG48|UBC_HUMAN Polyubiquitin-C OS=Homo sapiens OX=9606 GN=UBC PE
calmodulin   P0DP23  len= 149  >sp|P0DP23|CALM1_HUMAN Calmodulin-1 OS=Homo sapiens OX=9606 GN=CALM1 P


### Manually edit to get smaller chunk of ubiquitin

In [3]:
p = out / "ubiquitin.fasta"
seq = "".join(p.read_text().split("\n")[1:])
p.write_text(f">ubiquitin\n{seq[:76]}\n")
assert seq[:76].endswith("LRGG")  # canonical C-terminal tail

## Config

In [4]:
MODEL_NAME = "model_3_ptm"

# per-protein annotations for the viz; only calmodulin has lobes
ANNOTATIONS = {
    "calmodulin": {
        "N_LOBE":   (1, 75),
        "LINKER":   (76, 81),
        "C_LOBE":   (82, 148),
        "EF_HANDS": [(20, 31), (56, 67), (93, 104), (129, 140)],
    },
}

OPENFOLD_DIR = pathlib.Path(os.environ.get("OPENFOLD_DIR", "~/repos/openfold")).expanduser()
PARAMS_PATH  = OPENFOLD_DIR / "openfold" / "resources" / "params" / f"params_{MODEL_NAME}.npz"

WORK = pathlib.Path("./work").resolve()
for sub in ["fasta", "alignments", "out", "ref", "curves", "pair", "viz"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

NAMES = list(ACCESSIONS.keys())

SEQUENCES = {}
for name in NAMES:
    txt = (WORK / "fasta" / f"{name}.fasta").read_text()
    SEQUENCES[name] = "".join(txt.strip().split("\n")[1:])

print({k: len(v) for k, v in SEQUENCES.items()})
print("params exists:", PARAMS_PATH.exists())

{'myoglobin': 154, 'GFP': 238, 'TIM': 248, 'ubiquitin': 76, 'calmodulin': 149}
params exists: True


## Get MSA alignment for each protein

In [5]:
from colabfold.colabfold import run_mmseqs2

MSA_DEPTH = {}
(WORK / "mmseqs_cache").mkdir(parents=True, exist_ok=True)
for name in NAMES:
    (WORK / "alignments" / name).mkdir(parents=True, exist_ok=True)
    a3m_path = WORK / "alignments" / name / "mmseqs.a3m"

    if not a3m_path.exists():
        res = run_mmseqs2(
            SEQUENCES[name],
            prefix=str(WORK / "mmseqs_cache" / name),   # per-protein cache
            use_env=True, use_filter=True, use_templates=False,
        )
        a3m_path.write_text(res[0] if isinstance(res, (list, tuple)) else res)

    n = a3m_path.read_text().count(">")
    MSA_DEPTH[name] = n
    print(f"{name:12s} depth {n:6d}")
    assert n > 100, f"{name}: MSA suspiciously shallow"

myoglobin    depth   1331
GFP          depth    743
TIM          depth  16985
ubiquitin    depth  21577
calmodulin   depth  21884


## Load the model once

In [6]:
from openfold.config import model_config
from openfold.data import data_pipeline, feature_pipeline
from openfold.model.model import AlphaFold
from openfold.utils.import_weights import import_jax_weights_
from openfold.utils.tensor_utils import tensor_tree_map
from openfold.np import protein, residue_constants

config = model_config(MODEL_NAME)
N_BLK = config.model.evoformer_stack.no_blocks
C_Z   = config.model.evoformer_stack.c_z
print(f"blocks {N_BLK}  c_z {C_Z}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AlphaFold(config)
model.eval()
import_jax_weights_(model, str(PARAMS_PATH), version=MODEL_NAME)
model = model.to(device)

dp = data_pipeline.DataPipeline(template_featurizer=None)
fp = feature_pipeline.FeaturePipeline(config.data)

blocks 48  c_z 128


## Run the model for each protein

Two captures happen in the same forward pass.

**Full pair representations** at `SAVE_BLOCKS` only, for the channel browser later. These
are `(L, L, 128)` and expensive: at L=238 that is ~14 MB per block even in fp16, so only
8 of 48 blocks are kept.

**Reduced readouts** at *every* block and *every* recycle, for the scrubber. Each is
`(L, L)`, ~110 KB at L=238, so all 192 states cost about as much as two full pair reps.

The hooks append to lists rather than writing into a dict keyed by block index. That
matters: hooks fire once per block *per recycle*, so a dict would silently keep only the
last pass. The list is regrouped afterwards by finding where block 0 fires.

Two defensive patterns worth keeping permanently:

- `_forward_hooks.clear()` before registering. A hook left behind by a run that raised is
  never removed, and its closure keeps captured tensors alive
- `try/finally` around the whole thing, so a failure cannot leave hooks registered

In [7]:
from openfold.np import residue_constants as rc

### Distogram readout setup

The scrubber needs three `(L, L)` maps per block per recycle, all derived from the same
pair representation `z`:

| array | what it is | range |
|---|---|---|
| `pc` | probability mass below 8 A — P(contact) | 0 to 1 |
| `ed` | expected Cb-Cb distance | ~3 to 22 A |
| `zn` | `z.norm(dim=-1)` across the 128 channels | no fixed range |

`pc` and `ed` both pass through the distogram head, which was trained only on the final
block, so intermediate readouts are off-distribution. `zn` does not touch the head at all —
it is the panel that shows the pair rep changing regardless of whether the head can read it.

The 64th distogram bin is a catch-all absorbing everything past 22 A, which biases `ed`
upward. That is why contacts are defined from `pc` (probability mass) rather than
`ed < 8`.

In [11]:
# AF2 distogram: 64 bins spanning 2 to 22 A, last bin is a catch-all
bins  = torch.linspace(2.3125, 21.6875, 63, device=device)
edges = torch.cat([bins, bins[-1:] + 0.3125])
CONTACT_BIN = int((8.0 - 2.3125) / 0.3125) + 1
print(f"bins below 8 A: {CONTACT_BIN}  (upper edge {float(bins[CONTACT_BIN-1]):.3f} A)")

CB_IDX, CA_IDX = rc.atom_order["CB"], rc.atom_order["CA"]

def cb_contact_map(atom_positions, atom_mask, cutoff=8.0):
    """Cb-Cb contact map from predicted coordinates, Ca substituted where Cb is absent."""
    mask = np.asarray(atom_mask).astype(bool)
    cb = atom_positions[:, CB_IDX, :].copy()
    no_cb = ~mask[:, CB_IDX]
    cb[no_cb] = atom_positions[no_cb, CA_IDX, :]
    d = np.linalg.norm(cb[:, None, :] - cb[None, :, :], axis=-1)
    return (d < cutoff), d

bins below 8 A: 19  (upper edge 7.938 A)


In [12]:
NAMES

['myoglobin', 'GFP', 'TIM', 'ubiquitin', 'calmodulin']

In [13]:
SAVE_BLOCKS = list(range(0, N_BLK, 6))   # 8 of 48, full (L, L, 128) pair reps

pair_store = {}                          # block -> full z, final recycle only
pc_store, ed_store, zn_store = [], [], []  # (block, map) appended in firing order

def _mk_hook(i):
    def hook(mod, inp, outp):
        z = outp[1] if isinstance(outp, (list, tuple)) else outp
        with torch.no_grad():
            p = model.aux_heads.distogram(z).softmax(-1)
            p = (p + p.transpose(0, 1)) / 2
            pc_store.append((i, p[..., :CONTACT_BIN].sum(-1).cpu()))
            ed_store.append((i, (p * edges).sum(-1).cpu()))
            zn_store.append((i, z.norm(dim=-1).cpu()))
        if i in SAVE_BLOCKS:
            pair_store[i] = z.detach().half().cpu().numpy()   # overwritten each recycle
    return hook

def _regroup(store, n_blk, L):
    """Flat firing-order list -> (n_pass, n_blk, L, L). Drops any incomplete pass."""
    starts = [k for k in range(len(store)) if store[k][0] == 0]
    passes = [store[s:s + n_blk] for s in starts]
    passes = [p for p in passes
              if len(p) == n_blk and [b for b, _ in p] == list(range(n_blk))]
    if not passes:
        raise RuntimeError("no complete Evoformer pass captured")
    return torch.stack([t for p in passes for _, t in p]
                       ).reshape(len(passes), n_blk, L, L).numpy()

SUMMARY = {}
handles = []
try:
    for blk in model.evoformer.blocks:
        blk._forward_hooks.clear()
    handles = [blk.register_forward_hook(_mk_hook(i))
               for i, blk in enumerate(model.evoformer.blocks)]

    for name in NAMES[3:]:
        seq = SEQUENCES[name]
        L = len(seq)
        pair_store.clear()
        pc_store.clear(); ed_store.clear(); zn_store.clear()

        feature_dict = dp.process_fasta(
            fasta_path=str(WORK / "fasta" / f"{name}.fasta"),
            alignment_dir=str(WORK / "alignments" / name),
        )
        processed = fp.process_features(feature_dict, mode="predict")
        batch = {k: torch.as_tensor(v, device=device) for k, v in processed.items()}

        t0 = time.time()
        with torch.no_grad():
            out_t = model(batch)
        dt = time.time() - t0
        out = tensor_tree_map(lambda x: x.detach().cpu().numpy(), out_t)
        del out_t

        PC = _regroup(pc_store, N_BLK, L)
        ED = _regroup(ed_store, N_BLK, L)
        ZN = _regroup(zn_store, N_BLK, L)
        n_rec = PC.shape[0]

        # the head must reproduce the model's own output at the final block,
        # otherwise every intermediate readout is measuring something else
        p_ref = torch.as_tensor(out["distogram_logits"]).softmax(-1)
        p_ref = (p_ref + p_ref.transpose(0, 1)) / 2
        pc_ref = p_ref[..., :CONTACT_BIN].sum(-1).numpy()
        d_head = float(np.abs(PC[-1, -1] - pc_ref).max())
        assert d_head < 1e-3, f"{name}: head mismatch {d_head:.2e}"
        assert PC.var(axis=1).mean() > 0, f"{name}: no variation across blocks"

        plddt = out["plddt"]
        plddt_b = np.repeat(plddt[:, None], residue_constants.atom_type_num, axis=-1)

        batch_last = {k: v[..., -1].cpu().numpy() for k, v in batch.items()}
        unrelaxed = protein.from_prediction(
            features=batch_last, result=out, b_factors=plddt_b,
            remove_leading_feature_dimension=False,
        )
        pdb_path = WORK / "out" / f"{name}_{MODEL_NAME}.pdb"
        pdb_path.write_text(protein.to_pdb(unrelaxed))

        n_ca = sum(1 for l in pdb_path.read_text().splitlines()
                   if l.startswith("ATOM") and l[12:16].strip() == "CA")
        assert n_ca == L, f"{name}: expected {L} CA, got {n_ca}"

        BACKBONE = ["N", "CA", "C", "O"]
        resnames = np.array([rc.restype_1to3.get(rc.restypes[i], "UNK")
                             for i in batch_last["aatype"]])

        final_cm, final_d = cb_contact_map(out["final_atom_positions"],
                                           out["final_atom_mask"])

        np.savez_compressed(
            WORK / "out" / f"{name}_{MODEL_NAME}_scalars.npz",
            plddt=plddt.astype(np.float32),
            pae=out.get("predicted_aligned_error", np.zeros((L, L), np.float32)).astype(np.float32),
            ptm=np.asarray(out.get("ptm_score", np.nan), np.float32),
            msa=feature_dict["msa"].astype(np.uint8),
            deletion=feature_dict["deletion_matrix_int"].astype(np.uint8),
            seq=np.frombuffer(seq.encode(), np.uint8),
            atom_positions=out["final_atom_positions"].astype(np.float32),
            atom_mask=out["final_atom_mask"].astype(bool),
            aatype=batch_last["aatype"].astype(np.uint8),
            residue_index=batch_last["residue_index"].astype(np.int32),
            resname=resnames,
            bb_index=np.array([rc.atom_order[a] for a in BACKBONE]),
            bb_names=np.array(BACKBONE),
        )

        np.savez(
            WORK / "pair" / f"{name}_{MODEL_NAME}_z.npz",
            blocks=np.asarray(SAVE_BLOCKS),
            **{f"z_{i:02d}": pair_store[i] for i in SAVE_BLOCKS},
        )

        # --- everything the co-evolution scrubber needs, in one file -------------
        np.savez_compressed(
            WORK / "viz" / f"{name}_sweep.npz",
            pc=PC.astype(np.float16),          # (n_rec, n_blk, L, L)  0..1
            ed=ED.astype(np.float16),          # (n_rec, n_blk, L, L)  angstroms
            zn=ZN.astype(np.float16),          # (n_rec, n_blk, L, L)  unitless
            # global display scales: computed over the whole sweep so the viz does not
            # autoscale per frame and normalise away the growth it is meant to show
            zn_vmax=np.float32(np.percentile(ZN, 99.5)),
            ed_vmin=np.float32(3.0), ed_vmax=np.float32(22.0),
            n_rec=np.int32(n_rec), n_blk=np.int32(N_BLK), L=np.int32(L),
            seq=np.frombuffer(seq.encode(), np.uint8),
            resnum=batch_last["residue_index"].astype(np.int32) + 1,
            plddt=plddt.astype(np.float32),
            final_cm=final_cm,                 # (L, L) bool, from predicted coordinates
            final_d=final_d.astype(np.float16),
            msa_depth=np.int32(MSA_DEPTH[name]),
        )
        sweep_mb = (WORK / "viz" / f"{name}_sweep.npz").stat().st_size / 1e6

        SUMMARY[name] = dict(L=L, plddt=float(plddt.mean()), depth=MSA_DEPTH[name],
                             secs=round(dt, 1), n_rec=n_rec,
                             head_diff=d_head, sweep_mb=round(sweep_mb, 1))
        print(f"{name:12s} L={L:4d}  pLDDT {plddt.mean():5.1f}  {dt:5.1f}s  "
              f"{n_rec}x{N_BLK} states  sweep {sweep_mb:5.1f} MB")

        del batch, out, processed, feature_dict, PC, ED, ZN
        torch.cuda.empty_cache()
finally:
    for h in handles:
        h.remove()
    for blk in model.evoformer.blocks:
        blk._forward_hooks.clear()

ubiquitin    L=  76  pLDDT  95.5   13.6s  4x48 states  sweep   4.2 MB
calmodulin   L= 149  pLDDT  80.5   32.7s  4x48 states  sweep  16.9 MB


### Check what was written

Loads each sweep back and confirms the arrays vary along the block axis, that the ranges
are what the viz will assume, and that the recycle boundaries are where they should be.

A flat `pc.var(axis=1)` means the regrouping went wrong and every frame is identical —
which renders as a scrubber that does nothing.

In [15]:
for name in NAMES[3:]:
    p = WORK / "viz" / f"{name}_sweep.npz"
    d = np.load(p)
    pc, ed, zn = d["pc"].astype(np.float32), d["ed"].astype(np.float32), d["zn"].astype(np.float32)
    n_rec, n_blk = int(d["n_rec"]), int(d["n_blk"])
    print(f"{name:12s} {pc.shape}  {p.stat().st_size/1e6:5.1f} MB")
    print(f"             pc {pc.min():.3f}..{pc.max():.3f}   "
          f"ed {ed.min():.1f}..{ed.max():.1f} A   zn {zn.min():.0f}..{zn.max():.0f}")
    print(f"             var across blocks {pc.var(axis=1).mean():.5f}   "
          f"across recycles {pc.var(axis=0).mean():.5f}")
    print(f"             true contacts (final, |i-j|>12): "
          f"{int((d['final_cm'] & (np.abs(np.arange(pc.shape[-1])[:,None] - np.arange(pc.shape[-1])[None,:]) > 12)).sum()//2)}")
    assert pc.var(axis=1).mean() > 1e-6, f"{name}: frames identical — regrouping failed"

total = sum((WORK / "viz" / f"{n}_sweep.npz").stat().st_size for n in NAMES[3:]) / 1e6
print(f"\ntotal sweep data: {total:.1f} MB")
print("\nfor web delivery, quantise pc and ed to uint8 before shipping:")
print("  pc_u8 = (pc * 255).astype(np.uint8)              # 1/255 is plenty for display")
print("  ed_u8 = ((ed - 3) / 19 * 255).clip(0, 255).astype(np.uint8)")

ubiquitin    (4, 48, 76, 76)    4.2 MB
             pc 0.000..1.000   ed 2.3..22.0 A   zn 87..3812
             var across blocks 0.01619   across recycles 0.00111
             true contacts (final, |i-j|>12): 113
calmodulin   (4, 48, 149, 149)   16.9 MB
             pc 0.000..1.000   ed 2.3..22.0 A   zn 54..3982
             var across blocks 0.00749   across recycles 0.00059
             true contacts (final, |i-j|>12): 99

total sweep data: 21.0 MB

for web delivery, quantise pc and ed to uint8 before shipping:
  pc_u8 = (pc * 255).astype(np.uint8)              # 1/255 is plenty for display
  ed_u8 = ((ed - 3) / 19 * 255).clip(0, 255).astype(np.uint8)
